<a href="https://colab.research.google.com/github/mxrch5th/2026-Visionaries-Kiosk/blob/main/age_detection_ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


!pip install roboflow -q

from roboflow import Roboflow

# ⚠️ 아래 api_key를 본인 키로 교체하세요!
rf = Roboflow(api_key="3v2NsaCn9lgONw9a7907")
project = rf.workspace("sjh051217-gmail-com").project("age-detection-mgmcv")
version = project.version(2)
dataset = version.download("folder")

print("✅ 데이터셋 v2 다운로드 완료!")
print(f"데이터셋 경로: {dataset.location}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 16.4 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to age-detection-2 in folder:: 100%|██████████| 207/207 [00:00<00:00, 4966.60it/s]

✅ 데이터셋 v2 다운로드 완료!
데이터셋 경로: /content/age-detection-2


In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image

# GPU 대신 CPU 강제 사용 (CUDA 에러 방지)
device = torch.device("cpu")
print(f"✅ 사용 디바이스: {device}")

✅ 사용 디바이스: cpu


In [3]:
dataset_path = dataset.location
train_dir = os.path.join(dataset_path, "train")
valid_dir = os.path.join(dataset_path, "valid")
test_dir  = os.path.join(dataset_path, "test")

# 클래스 확인
all_classes = sorted(os.listdir(train_dir))
print("전체 클래스 목록:", all_classes)

# 이미지 수 확인
for split, path in [("Train", train_dir), ("Valid", valid_dir), ("Test", test_dir)]:
    if os.path.exists(path):
        for cls in os.listdir(path):
            cls_path = os.path.join(path, cls)
            if os.path.isdir(cls_path):
                count = len(os.listdir(cls_path))
                print(f"[{split}] {cls}: {count}장")

전체 클래스 목록: ['junior', 'senior', 'young']
[Train] junior: 42장
[Train] senior: 105장
[Train] young: 33장
[Valid] junior: 4장
[Valid] senior: 10장
[Valid] young: 3장
[Test] junior: 2장
[Test] senior: 5장
[Test] young: 1장


In [4]:
class BinaryAgeDataset(Dataset):
    """
    senior 폴더 → 라벨 1 (50대 이상)
    그 외 모든 폴더(young, 10대 등) → 라벨 0 (50대 미만)
    """
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.data = []
        for cls in os.listdir(root_dir):
            cls_path = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_path):
                continue
            label = 1 if cls.lower() == "senior" else 0
            for img_file in os.listdir(cls_path):
                if img_file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
                    self.data.append((os.path.join(cls_path, img_file), label))
        print(f"총 {len(self.data)}장 로드 완료")
        print(f"  senior(1): {sum(1 for _,l in self.data if l==1)}장")
        print(f"  young(0):  {sum(1 for _,l in self.data if l==0)}장")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        path, label = self.data[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

print("=== Train ===")
train_dataset = BinaryAgeDataset(train_dir, transform=train_transform)
print("=== Valid ===")
valid_dataset = BinaryAgeDataset(valid_dir, transform=valid_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False)

print("\n✅ 데이터 로더 준비 완료!")

=== Train ===
총 180장 로드 완료
  senior(1): 105장
  young(0):  75장
=== Valid ===
총 17장 로드 완료
  senior(1): 10장
  young(0):  7장

✅ 데이터 로더 준비 완료!


In [5]:
model = models.mobilenet_v2(pretrained=True)
model.classifier[1] = nn.Linear(model.last_channel, 2)  # 이진 분류 (senior / young)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✅ 모델 준비 완료!")

The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 61.3MB/s]

✅ 모델 준비 완료!


In [6]:
EPOCHS = 10
train_losses, valid_losses = [], []
train_accs,   valid_accs   = [], []
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- 학습 ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    train_losses.append(running_loss / len(train_loader))
    train_accs.append(100. * correct / total)

    # --- 검증 ---
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in valid_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            v_loss    += loss.item()
            _, predicted = outputs.max(1)
            v_correct += predicted.eq(labels).sum().item()
            v_total   += labels.size(0)
    valid_losses.append(v_loss / len(valid_loader))
    v_acc = 100. * v_correct / v_total
    valid_accs.append(v_acc)

    if v_acc > best_acc:
        best_acc = v_acc
        torch.save(model.state_dict(), "best_model_v2.pth")

    print(f"Epoch [{epoch+1:02d}/{EPOCHS}] "
          f"Train Loss: {train_losses[-1]:.4f} Acc: {train_accs[-1]:.1f}% | "
          f"Valid Loss: {valid_losses[-1]:.4f} Acc: {v_acc:.1f}%")

print(f"\n✅ 학습 완료! 최고 검증 정확도: {best_acc:.1f}%")

Epoch [01/10] Train Loss: 0.5316 Acc: 77.8% | Valid Loss: 0.9273 Acc: 70.6%
Epoch [02/10] Train Loss: 0.2285 Acc: 91.7% | Valid Loss: 0.2470 Acc: 82.4%
Epoch [03/10] Train Loss: 0.0767 Acc: 96.1% | Valid Loss: 0.0937 Acc: 88.2%
Epoch [04/10] Train Loss: 0.0867 Acc: 97.8% | Valid Loss: 0.1375 Acc: 94.1%
Epoch [05/10] Train Loss: 0.3572 Acc: 94.4% | Valid Loss: 0.1658 Acc: 82.4%
Epoch [06/10] Train Loss: 0.1356 Acc: 94.4% | Valid Loss: 0.6154 Acc: 76.5%
Epoch [07/10] Train Loss: 0.1389 Acc: 94.4% | Valid Loss: 0.1265 Acc: 94.1%
Epoch [08/10] Train Loss: 0.0749 Acc: 96.7% | Valid Loss: 0.1912 Acc: 94.1%
Epoch [09/10] Train Loss: 0.2428 Acc: 97.8% | Valid Loss: 0.2074 Acc: 82.4%
Epoch [10/10] Train Loss: 0.1589 Acc: 95.6% | Valid Loss: 0.3503 Acc: 88.2%

✅ 학습 완료! 최고 검증 정확도: 94.1%


In [8]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from PIL import Image
from google.colab import files
import sys

# =====================================================================
# 📂 [필수 단계] 구글 드라이브를 이 코랩 컴퓨터와 강제로 연결합니다.
# =====================================================================
from google.colab import drive
print("🔗 구글 드라이브 연결을 시작합니다. 팝업창이 뜨면 [구글 드라이브에 연결]을 눌러주세요.")
drive.mount('/content/drive')

# 저장할 구글 드라이브 폴더 자동 생성 (내 드라이브에 'kiosk_age_project' 폴더가 생김)
save_dir = '/content/drive/MyDrive/kiosk_age_project'
os.makedirs(save_dir, exist_ok=True)

drive_model_path = os.path.join(save_dir, 'best_model_v2.pth')
local_model_path = 'best_model_v2.pth'

# 1. 디바이스 및 모델 구조 설정
device = torch.device("cpu")
model = models.mobilenet_v2(pretrained=False)
model.classifier[1] = nn.Linear(model.last_channel, 2)
model = model.to(device)

# 2. 📂 가중치 로드 우선순위 (구글 드라이브에 누적된 게 있다면 무조건 그걸 1순위로 이어받음)
if os.path.exists(drive_model_path):
    model.load_state_dict(torch.load(drive_model_path, map_location=device))
    print(f"\n✨ [연동 대성공] 구글 드라이브에서 '기존 누적 학습된 마스터 파일'을 성공적으로 가져왔습니다!")
    print(f"   -> 경로: {drive_model_path}")
elif os.path.exists(local_model_path):
    model.load_state_dict(torch.load(local_model_path, map_location=device))
    print(f"\n🌱 [초기화 완료] 구글 드라이브에 저장된 파일이 없어, 방금 6단계에서 학습한 기본 가중치로 시작합니다.")
else:
    print("\n❌ 에러: 불러올 기본 모델(.pth) 파일이 없습니다. 6단계를 먼저 실행해 주세요.")
    sys.exit()

# 3. 누적 학습 설정 (기존 기억 보존용 아주 작은 학습률)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# =====================================================================
# 🚀 실시간 누적 학습 루프 실행
# =====================================================================
print("\n" + "="*50)
print("▶ 모델에 실시간으로 [누적 학습할 새 사진]을 업로드합니다.")
print("  (학습을 완전히 끝내려면 파일 선택 창에서 '취소(Cancel)'를 누르세요)")
print("="*50)

while True:
    uploaded = files.upload()
    if not uploaded:
        print("\n👋 사진 등록 및 누적 학습 시스템을 종료합니다.")
        break

    for original_name in uploaded.keys():
        try:
            model.eval()
            img = Image.open(original_name).convert("RGB")
            img_tensor = valid_transform(img).unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = model(img_tensor)
                probs = torch.softmax(outputs, dim=1)[0]
                pred = probs.argmax().item()

            current_label = "50대 이상 (senior)" if pred == 1 else "50대 미만 (young)"
            print(f"\n🔍 [현재 AI의 예측]: {current_label} (신뢰도: {probs[pred].item()*100:.1f}%)")

            print("💡 이 사진의 진짜 정답 라벨을 지정해 주세요.")
            print("   0: 50대 미만 (young)  /  1: 50대 이상 (senior)")
            answer = input("정답 번호 입력 (0 또는 1): ")
            while answer not in ['0', '1']:
                answer = input("잘못된 입력입니다. 0 또는 1만 입력하세요: ")

            correct_label = int(answer)

            # 🔥 실시간 역전파 수행
            model.train()
            optimizer.zero_grad()
            outputs = model(img_tensor)
            target_tensor = torch.tensor([correct_label]).to(device)
            loss = criterion(outputs, target_tensor)
            loss.backward()
            optimizer.step()

            print(f"✔️ 가중치 업데이트 성공! (Loss 오차값: {loss.item():.4f})")

            # 💾 덮어쓰기 저장 (코랩 임시 공간 + 구글 드라이브 동시 저장!)
            torch.save(model.state_dict(), local_model_path)
            torch.save(model.state_dict(), drive_model_path)
            print(f"💾 [클라우드 백업 완료] 구글 드라이브에 실시간으로 자동 덮어쓰기 완료! -> {drive_model_path}")

        except Exception as e:
            print(f"⚠️ 이미지 처리 중 오류 발생: {e}")
        finally:
            if os.path.exists(original_name):
                os.remove(original_name)
    print("-"*50)

🔗 구글 드라이브 연결을 시작합니다. 팝업창이 뜨면 [구글 드라이브에 연결]을 눌러주세요.
Mounted at /content/drive

🌱 [초기화 완료] 구글 드라이브에 저장된 파일이 없어, 방금 6단계에서 학습한 기본 가중치로 시작합니다.

▶ 모델에 실시간으로 [누적 학습할 새 사진]을 업로드합니다.
  (학습을 완전히 끝내려면 파일 선택 창에서 '취소(Cancel)'를 누르세요)


Saving KakaoTalk_20260618_184432653.jpg to KakaoTalk_20260618_184432653.jpg
Saving KakaoTalk_20260618_184432653_01.jpg to KakaoTalk_20260618_184432653_01.jpg
Saving KakaoTalk_20260618_184432653_02.jpg to KakaoTalk_20260618_184432653_02 (1).jpg
Saving KakaoTalk_20260618_184432653_03.jpg to KakaoTalk_20260618_184432653_03 (1).jpg
Saving KakaoTalk_20260618_184432653_04.jpg to KakaoTalk_20260618_184432653_04 (1).jpg
Saving KakaoTalk_20260618_184432653_05.jpg to KakaoTalk_20260618_184432653_05 (1).jpg
Saving KakaoTalk_20260618_184432653_06.webp to KakaoTalk_20260618_184432653_06 (1).webp
Saving KakaoTalk_20260618_184432653_07.webp to KakaoTalk_20260618_184432653_07 (1).webp
Saving KakaoTalk_20260618_184432653_08.webp to KakaoTalk_20260618_184432653_08 (1).webp
Saving KakaoTalk_20260618_184432653_09.jpg to KakaoTalk_20260618_184432653_09 (1).jpg

🔍 [현재 AI의 예측]: 50대 이상 (senior) (신뢰도: 99.9%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (senior)
정답 번호 입력 (0 또는 1): 0
✔️ 가중치 업데이트

Saving KakaoTalk_20260618_184432653_10.jpg to KakaoTalk_20260618_184432653_10.jpg

🔍 [현재 AI의 예측]: 50대 이상 (senior) (신뢰도: 100.0%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (senior)
정답 번호 입력 (0 또는 1): 1
✔️ 가중치 업데이트 성공! (Loss 오차값: 1.3310)
⚠️ 이미지 처리 중 오류 발생: Parent directory /content/drive/MyDrive/kiosk_age_project does not exist.
--------------------------------------------------


Saving KakaoTalk_20260618_184432653_11.jpg to KakaoTalk_20260618_184432653_11.jpg
Saving KakaoTalk_20260618_184432653_12.jpg to KakaoTalk_20260618_184432653_12.jpg
Saving KakaoTalk_20260618_184432653_13.jpg to KakaoTalk_20260618_184432653_13.jpg
Saving KakaoTalk_20260618_184432653_14.jpg to KakaoTalk_20260618_184432653_14.jpg
Saving KakaoTalk_20260618_184432653_15.png to KakaoTalk_20260618_184432653_15.png
Saving KakaoTalk_20260618_184432653_16.jpg to KakaoTalk_20260618_184432653_16.jpg

🔍 [현재 AI의 예측]: 50대 이상 (senior) (신뢰도: 100.0%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (senior)
정답 번호 입력 (0 또는 1): 0
✔️ 가중치 업데이트 성공! (Loss 오차값: 0.3818)
💾 [클라우드 백업 완료] 구글 드라이브에 실시간으로 자동 덮어쓰기 완료! -> /content/drive/MyDrive/kiosk_age_project/best_model_v2.pth

🔍 [현재 AI의 예측]: 50대 이상 (senior) (신뢰도: 92.6%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (senior)
정답 번호 입력 (0 또는 1): 0
✔️ 가중치 업데이트 성공! (Loss 오차값: 0.2381)
💾 [클라우드 백업 완료] 구글 드라이브에 실시간으로 자동 덮어쓰기 완료! -> /content/drive

Saving KakaoTalk_20260618_184432653_17.jpg to KakaoTalk_20260618_184432653_17.jpg
Saving KakaoTalk_20260618_184432653_18.jpg to KakaoTalk_20260618_184432653_18.jpg
Saving KakaoTalk_20260618_184432653_19.jpg to KakaoTalk_20260618_184432653_19.jpg
Saving KakaoTalk_20260618_184432653_20.jpg to KakaoTalk_20260618_184432653_20.jpg
Saving KakaoTalk_20260618_184432653_21.jpg to KakaoTalk_20260618_184432653_21.jpg
Saving KakaoTalk_20260618_184432653_22.jpg to KakaoTalk_20260618_184432653_22.jpg
Saving KakaoTalk_20260618_184432653_23.jpg to KakaoTalk_20260618_184432653_23.jpg
Saving KakaoTalk_20260618_184432653_24.jpg to KakaoTalk_20260618_184432653_24.jpg
Saving KakaoTalk_20260618_184432653_25.jpg to KakaoTalk_20260618_184432653_25.jpg
Saving KakaoTalk_20260618_184432653_26.jpg to KakaoTalk_20260618_184432653_26.jpg
Saving KakaoTalk_20260618_184432653_27.jpg to KakaoTalk_20260618_184432653_27.jpg
Saving KakaoTalk_20260618_184432653_28.jpg to KakaoTalk_20260618_184432653_28.jpg
Saving KakaoTalk

Saving KakaoTalk_20260618_184442210_07.jpg to KakaoTalk_20260618_184442210_07.jpg
Saving KakaoTalk_20260618_184442210_08 (1).jpg to KakaoTalk_20260618_184442210_08 (1).jpg
Saving KakaoTalk_20260618_184442210_09 (1).jpg to KakaoTalk_20260618_184442210_09 (1).jpg
Saving KakaoTalk_20260618_184442210_10.jpg to KakaoTalk_20260618_184442210_10.jpg

🔍 [현재 AI의 예측]: 50대 이상 (senior) (신뢰도: 100.0%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (senior)
정답 번호 입력 (0 또는 1): 0
✔️ 가중치 업데이트 성공! (Loss 오차값: 0.6936)
💾 [클라우드 백업 완료] 구글 드라이브에 실시간으로 자동 덮어쓰기 완료! -> /content/drive/MyDrive/kiosk_age_project/best_model_v2.pth

🔍 [현재 AI의 예측]: 50대 미만 (young) (신뢰도: 59.8%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (senior)
정답 번호 입력 (0 또는 1): 0
✔️ 가중치 업데이트 성공! (Loss 오차값: 0.8100)
💾 [클라우드 백업 완료] 구글 드라이브에 실시간으로 자동 덮어쓰기 완료! -> /content/drive/MyDrive/kiosk_age_project/best_model_v2.pth

🔍 [현재 AI의 예측]: 50대 미만 (young) (신뢰도: 100.0%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 

Saving KakaoTalk_20260618_184442210_11 (1).jpg to KakaoTalk_20260618_184442210_11 (1).jpg
Saving KakaoTalk_20260618_184442210_12 (1).jpg to KakaoTalk_20260618_184442210_12 (1).jpg
Saving KakaoTalk_20260618_184442210_13 (1).jpg to KakaoTalk_20260618_184442210_13 (1).jpg
Saving KakaoTalk_20260618_184442210_14 (1).jpg to KakaoTalk_20260618_184442210_14 (1).jpg
Saving KakaoTalk_20260618_184442210_15 (1).jpg to KakaoTalk_20260618_184442210_15 (1).jpg
Saving KakaoTalk_20260618_184442210_16 (1).jpg to KakaoTalk_20260618_184442210_16 (1).jpg
Saving KakaoTalk_20260618_184442210_19 (1).jpg to KakaoTalk_20260618_184442210_19 (1).jpg
Saving KakaoTalk_20260618_184442210_20 (1).jpg to KakaoTalk_20260618_184442210_20 (1).jpg
Saving KakaoTalk_20260618_184442210_23 (1).jpg to KakaoTalk_20260618_184442210_23 (1).jpg
Saving KakaoTalk_20260618_184442210_25.png to KakaoTalk_20260618_184442210_25.png

🔍 [현재 AI의 예측]: 50대 이상 (senior) (신뢰도: 97.7%)
💡 이 사진의 진짜 정답 라벨을 지정해 주세요.
   0: 50대 미만 (young)  /  1: 50대 이상 (

Saving KakaoTalk_20260618_184442210_18.jpg to KakaoTalk_20260618_184442210_18.jpg
Saving KakaoTalk_20260618_184442210_21.jpg to KakaoTalk_20260618_184442210_21.jpg
Saving KakaoTalk_20260618_184442210_22 (1).jpg to KakaoTalk_20260618_184442210_22 (1).jpg
Saving KakaoTalk_20260618_184442210_24.png to KakaoTalk_20260618_184442210_24.png
Saving KakaoTalk_20260618_184442210_26 (1).jpg to KakaoTalk_20260618_184442210_26 (1).jpg
Saving KakaoTalk_20260618_184442210_27 (1).jpg to KakaoTalk_20260618_184442210_27 (1).jpg
Saving KakaoTalk_20260618_184442210_28 (1).png to KakaoTalk_20260618_184442210_28 (1).png
Saving KakaoTalk_20260618_184442210_29 (1).jpg to KakaoTalk_20260618_184442210_29 (1).jpg
Saving KakaoTalk_20260618_184446888.jpg to KakaoTalk_20260618_184446888.jpg
Saving KakaoTalk_20260618_184446888_01.jpg to KakaoTalk_20260618_184446888_01.jpg
Saving KakaoTalk_20260618_184446888_03.jpg to KakaoTalk_20260618_184446888_03.jpg

🔍 [현재 AI의 예측]: 50대 미만 (young) (신뢰도: 100.0%)
💡 이 사진의 진짜 정답 라벨을 지


👋 사진 등록 및 누적 학습 시스템을 종료합니다.
